# Report + Business Stakeholder Review — Formatters Review

Human-run notebook for the **pure-formatter slice** of Sean Step 6 (Report + Business Stakeholder Review).

Plan: `project_planning/sean_step_artifacts/Report_Business_Review_Implementation_Plan.md`  
Checklist: `project_planning/sean_step_artifacts/Report_Business_Review_Checklist.md`

Run cells top to bottom to:
- inspect the four pure formatters added to `tools/reporting.py`
- see how each one renders state slices into prompt-context Markdown
- confirm the formatters take plain mappings (no `PipelineState` import) and return strings only

These functions are what the `report_writer` agent calls before the LLM — they own the state→prompt translation.

In [ ]:
from pathlib import Path
import inspect
import subprocess

from multi_agent_ds.tools.reporting import (
    format_data_summary,
    format_modeling_summary,
    format_evaluation_summary,
    format_decision_trace,
)

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find the repo root.")

ROOT = resolve_repo_root()
print("Repo root:", ROOT)

def run_pytest(args: list[str]) -> None:
    cmd = ["uv", "run", "pytest", *args]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=ROOT)
    if completed.returncode != 0:
        raise RuntimeError(f"pytest failed with exit code {completed.returncode}")


## 1. Confirm import-direction discipline

`tools/reporting.py` must not import from `agents/`, `orchestration/`, or `workflows/`. The check below looks at the module source.

**Human review questions:**
- Are all imports either stdlib or `multi_agent_ds.core.*` (no `agents` / `orchestration` / `workflows`)?
- Are the formatters typed against `Mapping` / `Sequence` rather than `PipelineState`?

In [ ]:
import multi_agent_ds.tools.reporting as reporting_module

src = inspect.getsource(reporting_module)
import_lines = [line for line in src.splitlines() if line.startswith(("import ", "from "))]
for line in import_lines:
    print(line)

forbidden = ("multi_agent_ds.agents", "multi_agent_ds.orchestration", "multi_agent_ds.workflows")
violations = [line for line in import_lines if any(f in line for f in forbidden)]
assert not violations, f"tools/reporting.py imports from forbidden layer: {violations}"
print("\nNo forbidden imports — formatters stay pure.")


## 2. `format_data_summary`

Slices a `data_summary` dict into a Markdown bullet list of train/val/test counts, feature counts, target rate, and ground-truth flag.

**Human review question:** does this match what a non-technical stakeholder needs to see at the top of the report?

In [ ]:
data_summary = {
    "n_train": 700,
    "n_validation": 150,
    "n_test": 150,
    "n_features": 9,
    "n_numerical": 6,
    "n_categorical": 3,
    "target_rate_train": 0.183456,
    "target_rate_test": 0.179012,
    "has_ground_truth": True,
}
print(format_data_summary(data_summary))
print("\n--- empty payload fallback ---")
print(format_data_summary({}))


## 3. `format_modeling_summary`

Renders the final `ModelingVerdict` plus the per-algorithm final-phase pointer. This is the core block the writer leans on for "which model won and why."

In [ ]:
verdict = {
    "summary": "LightGBM leads on every metric after tuning.",
    "best_algorithm": "lightgbm",
    "ranked_algorithms": ["lightgbm", "logistic_regression"],
    "final_metrics": {
        "lightgbm": {"gini": 0.66, "roc_auc": 0.83},
        "logistic_regression": {"gini": 0.51, "roc_auc": 0.74},
    },
    "justification": "Lead larger than either model's CV std.",
    "next_action": "proceed_to_evaluation",
}
modeling_results = {
    "final_candidates": {
        "lightgbm": {"final_phase": "feature_selection"},
        "logistic_regression": {"final_phase": "baseline"},
    }
}
print(format_modeling_summary(verdict, modeling_results))
print("\n--- missing verdict fallback ---")
print(format_modeling_summary(None))


## 4. `format_evaluation_summary`

Reads Jonathan's `evaluation_result` payload (Step 9). The agent treats this slot as optional, so the formatter gives a clear caveat block when the slot is missing or empty.

**Human review question:** when evaluation is missing, is the caveat phrased honestly without scaring a stakeholder?

In [ ]:
evaluation_result = {
    "winner": "lightgbm",
    "primary_metric": "gini",
    "rankings": {"gini": ["lightgbm", "logistic_regression"], "roc_auc": ["lightgbm", "logistic_regression"]},
    "ground_truth_comparison": {
        "lightgbm": {"mse_vs_true_prob": 0.001234},
        "logistic_regression": {"mse_vs_true_prob": 0.004567},
    },
    "shap": {"top_features": ["age", "income", "credit_score", "region", "tenure"]},
    "notes": "LightGBM tracks the Bayes-optimal probability tightly on numerical features.",
}
print(format_evaluation_summary(evaluation_result))
print("\n--- missing evaluation fallback (the common case while Jonathan Step 2 is unbuilt) ---")
print(format_evaluation_summary(None))


## 5. `format_decision_trace`

Compresses the `agent_decisions` log into the most recent N entries. Default cap is 25 — enough to show the recent loop without flooding the prompt with the entire history.

**Human review question:** is 25 entries the right default? Modeling can record many entries per phase; if this feels long, drop it in `tools/reporting.py`.

In [ ]:
trace = [
    {"agent": "ml_modeler", "phase": "baseline", "summary": "Both algos worth tuning", "algorithms_to_tune": ["lightgbm", "logistic_regression"]},
    {"agent": "ml_reviewer", "phase": "baseline_review", "approved": True, "summary": "Decision sound"},
    {"agent": "ml_modeler", "phase": "tune", "algorithm": "lightgbm", "reasoning": "Improvement above noise"},
    {"agent": "ml_modeler", "phase": "final_recommendation", "best_algorithm": "lightgbm", "next_action": "proceed_to_evaluation"},
    {"agent": "ml_reviewer", "phase": "final_recommendation_review", "approved": True, "summary": "Verdict holds"},
]
print(format_decision_trace(trace))
print("\n--- empty trace fallback ---")
print(format_decision_trace(None))


## 6. Run the formatter tests

These live in `tests/test_report_business_review.py` and cover happy paths plus empty / missing payloads.

In [ ]:
run_pytest([
    "tests/test_report_business_review.py",
    "-k",
    "format_",
    "-v",
])


## 7. Sign-off

If the formatter outputs above are clear and grounded in real state, tick the **Step 2 — formatters** human-review boxes in `Report_Business_Review_Checklist.md`.